In [ ]:
!pip install -U transformers datasets evaluate accelerate torch

In [ ]:
import os
import json
import time
import platform
import socket
import subprocess
from datetime import datetime

import requests
import torch
from transformers import AutoTokenizer, AutoModel

In [ ]:

# ---------- Helpers ----------
def run(cmd):
    return subprocess.check_output(cmd, text=True).strip()

def get_imds_token(timeout=2):
    """Get IMDSv2 token (works when IMDS is enabled; required on many EC2 configs)."""
    try:
        r = requests.put(
            "http://169.254.169.254/latest/api/token",
            headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
            timeout=timeout,
        )
        if r.status_code == 200:
            return r.text
    except Exception:
        pass
    return None

def get_imds(path, token=None, timeout=2):
    """Query EC2 instance metadata. Returns None if not on EC2 / blocked."""
    url = f"http://169.254.169.254/latest/meta-data/{path}"
    headers = {}
    if token:
        headers["X-aws-ec2-metadata-token"] = token
    try:
        r = requests.get(url, headers=headers, timeout=timeout)
        if r.status_code == 200:
            return r.text
    except Exception:
        pass
    return None

In [ ]:
token = get_imds_token()
instance_id = get_imds("instance-id", token)
instance_type = get_imds("instance-type", token)
az = get_imds("placement/availability-zone", token)

print("=== EC2 Proof ===")
print("hostname:", socket.gethostname())
print("local time:", datetime.now().isoformat(timespec="seconds"))
print("instance-id:", instance_id)
print("instance-type:", instance_type)
print("availability-zone:", az)

if instance_id is None:
    print("WARNING: Could not access EC2 metadata. You might not be on EC2, or IMDS is disabled/blocked.")
else:
    print("OK: EC2 metadata accessible (strong proof you're on an EC2 instance).")

# Optional extra evidence: show OS + kernel
print("\n=== System ===")
print("platform:", platform.platform())
try:
    print("uname -a:", run(["uname", "-a"]))
except Exception:
    pass

In [ ]:
print("\n=== Python & Packages ===")
print("python:", platform.python_version())
print("torch:", torch.__version__)
print("transformers:", __import__("transformers").__version__)
print("requests:", requests.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda version:", torch.version.cuda)
    print("gpu name:", torch.cuda.get_device_name(0))

In [ ]:
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "hostname": socket.gethostname(),
    "instance_id": instance_id,
    "instance_type": instance_type,
    "availability_zone": az,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": __import__("transformers").__version__,
    "cuda_available": torch.cuda.is_available()
}

path = os.path.join(out_dir, "ec2_bert_smoketest.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print("\nSaved proof file:", path)
print("Done ✅")